# Day 11 / 42: Scaling and Encoding
### 42 Days of ML Challenge | @VaishnaviJagtap18

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week2_feature_work/day11_scaling_encoding/day11_notebook.ipynb)

---

## What You Will Learn
- Why sklearn's KNN and SVM break without scaling (with proof)
- MinMaxScaler vs StandardScaler: when to use which
- Label Encoding vs One-Hot Encoding vs Ordinal Encoding
- How to build a clean preprocessing pipeline
- The most common mistake that silently tanks model accuracy

---

## Step 0: Install and Import

In [ ]:
# Run this cell first - installs everything you need
!pip install scikit-learn pandas numpy matplotlib seaborn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    LabelEncoder,
    OrdinalEncoder,
    OneHotEncoder
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

np.random.seed(42)
print("All imports successful. You are ready for Day 11.")

---
## Step 1: Build a Realistic Dataset

We use a customer purchase prediction dataset with mixed feature types:
- **Numerical**: age (22-60), income (20k-150k) — very different scales
- **Categorical nominal**: city (Mumbai, Delhi, Bangalore) — no order
- **Categorical ordinal**: education level (high school < bachelors < masters) — has order
- **Target**: purchased (0 or 1)

In [ ]:
np.random.seed(42)
n = 300

age    = np.random.randint(22, 60, n).astype(float)
income = np.random.randint(20000, 150000, n).astype(float)
education = np.random.choice(['high_school', 'bachelors', 'masters'], n)
city      = np.random.choice(['Mumbai', 'Delhi', 'Bangalore'], n)

# Purchase rule: high income OR young customer is more likely to purchase
purchased = ((income > 80000) | (age < 30)).astype(int)

df = pd.DataFrame({
    'age':       age,
    'income':    income,
    'education': education,
    'city':      city,
    'purchased': purchased
})

print("Dataset shape:", df.shape)
print("\nFirst 8 rows:")
print(df.head(8).to_string(index=False))
print("\nData types:")
print(df.dtypes)
print("\nTarget distribution (0=no purchase, 1=purchase):")
print(df['purchased'].value_counts())

---
## Step 2: The Problem — Why Scale at All?

**Age** ranges from 22 to 60. Range = 38.  
**Income** ranges from 20,000 to 150,000. Range = 130,000.

KNN calculates Euclidean distance between points to find neighbors.  
With these raw values, income completely dominates the distance calculation.  
A difference of 1 year in age is invisible next to a difference of 1000 in income.

**The model ignores age entirely. Not because age is unimportant. Because income's scale drowns it out.**

Let's prove this with numbers.

In [ ]:
X = df[['age', 'income']].values
y = df['purchased'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# KNN WITHOUT scaling
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)
acc_raw = accuracy_score(y_test, knn_raw.predict(X_test))

# KNN WITH StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # transform only — never fit on test data

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
acc_scaled = accuracy_score(y_test, knn_scaled.predict(X_test_scaled))

print("KNN accuracy WITHOUT scaling: {:.1f}%".format(acc_raw * 100))
print("KNN accuracy WITH StandardScaler: {:.1f}%".format(acc_scaled * 100))
print("Improvement: {:.1f} percentage points".format((acc_scaled - acc_raw) * 100))
print()
print("Same model. Same data. Same k=5. Only difference: scaling.")
print("This is why scaling is not optional for distance-based algorithms.")

---
## Step 3: MinMaxScaler vs StandardScaler

| | MinMaxScaler | StandardScaler |
|---|---|---|
| **Output range** | [0, 1] | mean=0, std=1 |
| **Formula** | (x - min) / (max - min) | (x - mean) / std |
| **Sensitive to outliers?** | Yes — one extreme value compresses all others | Less — uses mean and std |
| **Use when** | Neural networks, image pixel values, bounded inputs | Most ML algorithms, when distribution matters |
| **Does NOT work well when** | Outliers are present in the column | Distribution is heavily skewed |

**Rule of thumb:** Default to StandardScaler. Switch to MinMaxScaler only for neural network inputs or when the feature is naturally bounded (e.g., percentage values).

In [ ]:
income_sample = df['income'].values.reshape(-1, 1)

mm_scaler = MinMaxScaler()
ss_scaler  = StandardScaler()

income_mm = mm_scaler.fit_transform(income_sample)
income_ss = ss_scaler.fit_transform(income_sample)

print("=== Original Income ===")
print(f"  Min:  {income_sample.min():>10.0f}")
print(f"  Max:  {income_sample.max():>10.0f}")
print(f"  Mean: {income_sample.mean():>10.0f}")
print(f"  Std:  {income_sample.std():>10.0f}")

print("\n=== After MinMaxScaler ===")
print(f"  Min:  {income_mm.min():>10.4f}")
print(f"  Max:  {income_mm.max():>10.4f}")
print(f"  Mean: {income_mm.mean():>10.4f}")

print("\n=== After StandardScaler ===")
print(f"  Min:  {income_ss.min():>10.4f}")
print(f"  Max:  {income_ss.max():>10.4f}")
print(f"  Mean: {income_ss.mean():>10.4f}")
print(f"  Std:  {income_ss.std():>10.4f}")

# Visualise the distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Effect of Scaling on Income Distribution', fontsize=13, fontweight='bold')

axes[0].hist(income_sample, bins=25, color='steelblue', edgecolor='white')
axes[0].set_title('Original Income', fontsize=11)
axes[0].set_xlabel('Value')

axes[1].hist(income_mm, bins=25, color='darkorange', edgecolor='white')
axes[1].set_title('MinMaxScaler (range: 0 to 1)', fontsize=11)
axes[1].set_xlabel('Value')

axes[2].hist(income_ss, bins=25, color='seagreen', edgecolor='white')
axes[2].set_title('StandardScaler (mean=0, std=1)', fontsize=11)
axes[2].set_xlabel('Value')

plt.tight_layout()
plt.savefig('day11_scaling_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Note: The SHAPE of the distribution does not change. Only the scale does.")

---
## Step 4: How Outliers Break MinMaxScaler

This is the most common mistake with MinMaxScaler in production.  
One extreme value (an outlier) shifts the min or max, which compresses all other values into a tiny range.

In [ ]:
normal_income = np.array([30000, 45000, 60000, 75000, 90000], dtype=float)

# Add one outlier — a CEO-level salary in a dataset of regular employees
income_with_outlier = np.array([30000, 45000, 60000, 75000, 90000, 5000000], dtype=float)

mm = MinMaxScaler()

scaled_normal  = mm.fit_transform(normal_income.reshape(-1,1)).flatten()
scaled_outlier = mm.fit_transform(income_with_outlier.reshape(-1,1)).flatten()

print("=== Without Outlier ===")
for orig, scaled in zip(normal_income, scaled_normal):
    print(f"  Income: {orig:>8.0f}  =>  MinMax: {scaled:.4f}")

print("\n=== With Outlier (5,000,000 added) ===")
for orig, scaled in zip(income_with_outlier, scaled_outlier):
    label = " <-- OUTLIER" if orig == 5000000 else ""
    print(f"  Income: {orig:>8.0f}  =>  MinMax: {scaled:.4f}{label}")

print()
print("The first 5 values get compressed into the range 0.005 to 0.017.")
print("The model sees almost no difference between a 30k and 90k salary.")
print("This is why MinMaxScaler fails silently when outliers are present.")

---
## Step 5: Encoding Categorical Features

Models work with numbers. Categorical text columns must be converted before training.  
There are 3 encoding strategies and choosing the wrong one silently breaks your model.

| Encoding | When to use | Risk if misused |
|---|---|---|
| **Label Encoding** | Only for the target variable (y) | Tells linear models Mumbai=2 > Delhi=1 > Bangalore=0 (false!) |
| **One-Hot Encoding** | Nominal features (no order) for linear models | Creates too many columns on high-cardinality features |
| **Ordinal Encoding** | Features with a real order (small < medium < large) | Must define the correct order manually |

In [ ]:
# === LABEL ENCODING ===
# Correct use: encoding the target variable y
# Wrong use: encoding a nominal feature like city

le = LabelEncoder()

# Encoding the target (correct)
target_raw = ['yes', 'no', 'yes', 'no', 'yes']
target_encoded = le.fit_transform(target_raw)
print("=== Label Encoding (correct — used on target) ===")
for raw, enc in zip(target_raw, target_encoded):
    print(f"  {raw:>5} => {enc}")
print()

# Encoding city (wrong — implies Bangalore < Delhi < Mumbai numerically)
city_sample = ['Mumbai', 'Delhi', 'Bangalore', 'Mumbai', 'Delhi']
city_label_encoded = le.fit_transform(city_sample)
print("=== Label Encoding on city (WRONG for linear models) ===")
for raw, enc in zip(city_sample, city_label_encoded):
    print(f"  {raw:>12} => {enc}")
print()
print("Problem: A linear model interprets Mumbai(2) as numerically twice Delhi(1).")
print("There is no such relationship in reality.")

In [ ]:
# === ONE-HOT ENCODING ===
# Use for nominal categorical features (no natural order)

city_df = pd.DataFrame({'city': ['Mumbai', 'Delhi', 'Bangalore', 'Mumbai', 'Delhi', 'Bangalore']})

# pandas get_dummies — easiest for prototyping
city_ohe = pd.get_dummies(city_df['city'], prefix='city')

print("=== One-Hot Encoding (city) ===")
result = pd.concat([city_df, city_ohe], axis=1)
print(result.to_string(index=False))
print()
print("Each city becomes its own binary column.")
print("No numeric ordering implied. The model sees each city independently.")
print()

# The dummy variable trap
print("=== The Dummy Variable Trap ===")
print("If Bangalore=0 and Delhi=0, then Mumbai must be 1.")
print("So one column is always redundant. For linear models, drop one column.")
city_ohe_dropped = pd.get_dummies(city_df['city'], prefix='city', drop_first=True)
result2 = pd.concat([city_df, city_ohe_dropped], axis=1)
print(result2.to_string(index=False))
print()
print("drop_first=True removes the redundant column.")
print("Tree-based models (Random Forest, XGBoost) do NOT need this — skip drop_first.")

In [ ]:
# === ORDINAL ENCODING ===
# Use when the feature has a meaningful order
# You MUST define the correct order manually

edu_df = pd.DataFrame({'education': ['bachelors', 'masters', 'high_school', 'masters', 'bachelors']})

# Define the correct order — this is domain knowledge, not guesswork
oe = OrdinalEncoder(categories=[['high_school', 'bachelors', 'masters']])
edu_df['education_encoded'] = oe.fit_transform(edu_df[['education']])

print("=== Ordinal Encoding (education) ===")
print(edu_df.to_string(index=False))
print()
print("The order is: high_school=0 < bachelors=1 < masters=2")
print("The model can correctly learn that more education correlates with higher value.")
print()
print("What happens if you get the order wrong?")
oe_wrong = OrdinalEncoder(categories=[['masters', 'high_school', 'bachelors']])
edu_df['education_wrong'] = oe_wrong.fit_transform(edu_df[['education']])
print(edu_df.to_string(index=False))
print()
print("Now masters=0 and bachelors=2. The model learns the exact opposite relationship.")
print("This is a silent error — no exception is thrown. Your model just learns wrong.")

---
## Step 6: The Production Pattern — ColumnTransformer + Pipeline

In production you never manually apply scalers and encoders step by step.  
That approach causes **data leakage** (fitting the scaler on test data) and **errors in deployment**.

The correct pattern: wrap everything in a `ColumnTransformer` inside a `Pipeline`.  
The pipeline handles the fit/transform split automatically and can be saved and reloaded as a single object.

**This is what production ML code actually looks like.**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Define columns by type
numerical_cols  = ['age', 'income']
nominal_cols    = ['city']           # no order — use OHE
ordinal_cols    = ['education']      # has order — use OrdinalEncoder

# Education order (must be defined explicitly)
edu_order = ['high_school', 'bachelors', 'masters']

# Build the preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('nom', OneHotEncoder(drop='first', sparse_output=False), nominal_cols),
        ('ord', OrdinalEncoder(categories=[edu_order]), ordinal_cols),
    ],
    remainder='drop'
)

# Full pipeline: preprocessing + model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', KNeighborsClassifier(n_neighbors=5))
])

# Train and evaluate
X_all = df[numerical_cols + nominal_cols + ordinal_cols]
y_all = df['purchased'].values

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

# One call handles fit + transform + model training
pipeline.fit(X_train_p, y_train_p)
acc_pipeline = accuracy_score(y_test_p, pipeline.predict(X_test_p))

print("Pipeline trained successfully.")
print(f"Accuracy with full pipeline (age + income + city + education): {acc_pipeline*100:.1f}%")
print()
print("Feature columns after preprocessing:")
ohe_cols = pipeline.named_steps['preprocessor'].named_transformers_['nom'].get_feature_names_out(['city'])
all_features = numerical_cols + list(ohe_cols) + ordinal_cols
for f in all_features:
    print(f"  {f}")

In [ ]:
import joblib

# Save the entire pipeline — scaler, encoder, and model in one file
joblib.dump(pipeline, 'day11_pipeline.pkl')
print("Pipeline saved to day11_pipeline.pkl")

# Load and use it on new data
loaded_pipeline = joblib.load('day11_pipeline.pkl')

new_customer = pd.DataFrame({
    'age':       [27],
    'income':    [95000],
    'city':      ['Mumbai'],
    'education': ['masters']
})

prediction = loaded_pipeline.predict(new_customer)
probability = loaded_pipeline.predict_proba(new_customer)

print()
print("=== Prediction on new customer ===")
print(f"  Age: 27 | Income: 95,000 | City: Mumbai | Education: Masters")
print(f"  Prediction: {'Will Purchase' if prediction[0]==1 else 'Will NOT Purchase'}")
print(f"  Probability: No Purchase={probability[0][0]:.2f}, Purchase={probability[0][1]:.2f}")
print()
print("This is how a real ML API serves predictions.")
print("The loaded pipeline applies identical preprocessing automatically.")
print("You never have to manually scale or encode incoming requests.")

---
## Step 7: The Real-World Production Problem

This is the mistake that costs teams days of debugging.

**The scenario:** A data scientist scales training data, trains a model, exports the model weights.  
In deployment, incoming requests are fed raw (unscaled) into the model.  
Predictions look correct in testing. In production they are silently wrong.

The fix is using a Pipeline — the scaler is bundled with the model.  
You save one object. You load one object. The preprocessing is never accidentally skipped.

Below: proof that raw data fed into a scaler-trained model produces wrong predictions.

In [ ]:
# Simulate the deployment bug

# Step 1: Train correctly (with scaler)
scaler_train = StandardScaler()
X_train_s = scaler_train.fit_transform(X_train)   # fit on training data
X_test_s  = scaler_train.transform(X_test)         # transform test data

model_correct = KNeighborsClassifier(n_neighbors=5)
model_correct.fit(X_train_s, y_train)

# Correct deployment: scale incoming data before prediction
test_input_raw = X_test[:5]
test_input_scaled = scaler_train.transform(test_input_raw)
correct_preds = model_correct.predict(test_input_scaled)

# Bug: raw data fed directly to the scaled model
buggy_preds = model_correct.predict(test_input_raw)

print("=== Deployment Bug: Raw Data vs Scaled Data ===")
print(f"{'Age':>6} {'Income':>10} | {'Correct':>10} | {'Buggy':>8}")
print("-" * 42)
for i in range(5):
    age_val = test_input_raw[i][0]
    inc_val = test_input_raw[i][1]
    match = "SAME" if correct_preds[i] == buggy_preds[i] else "DIFFERENT"
    print(f"  {age_val:>4.0f}   {inc_val:>8.0f} |        {correct_preds[i]} |        {buggy_preds[i]}  ({match})")

print()
print("Fix: Always save the scaler with the model using a Pipeline or joblib.")
print("Never apply the scaler manually in the serving code.")

---
## Step 8: Summary — Rules to Live By

In [ ]:
print("=" * 60)
print("DAY 11 SUMMARY: Scaling and Encoding Rules")
print("=" * 60)
print()
print("SCALING")
print("-" * 40)
print("1. Always scale for distance-based models (KNN, SVM, K-Means)")
print("2. Always scale for gradient-based models (Linear/Logistic Reg, Neural Nets)")
print("3. Tree-based models (Decision Tree, Random Forest, XGBoost) do NOT need scaling")
print("4. Default choice: StandardScaler")
print("5. Use MinMaxScaler only for neural nets or bounded inputs (no outliers)")
print("6. Fit the scaler on TRAINING data only. Transform test data separately.")
print()
print("ENCODING")
print("-" * 40)
print("7. Nominal features (city, color, brand): use OneHotEncoder")
print("8. Ordinal features (small/medium/large, education level): use OrdinalEncoder")
print("9. Target variable (y): LabelEncoder is fine")
print("10. Never use LabelEncoder on nominal input features for linear models")
print("11. High-cardinality nominals (>50 unique values): consider target encoding or hashing")
print()
print("PRODUCTION")
print("-" * 40)
print("12. Always use Pipeline — scaler and model saved together")
print("13. Never fit the scaler or encoder on test data")
print("14. Use ColumnTransformer to handle mixed feature types cleanly")
print()
print("=" * 60)

---
## Practice Exercise

Use the dataset below. It has mixed feature types and intentional encoding mistakes.

Your tasks:
1. Identify the correct encoding strategy for each feature
2. Build a `ColumnTransformer` that handles all columns correctly
3. Wrap it in a `Pipeline` with a KNN classifier
4. Train and report accuracy
5. Save the pipeline and load it to make a prediction on one new row

In [ ]:
# Practice dataset — loan approval prediction
np.random.seed(99)
n_practice = 250

practice_df = pd.DataFrame({
    'credit_score':   np.random.randint(300, 850, n_practice).astype(float),  # numerical
    'annual_income':  np.random.randint(25000, 200000, n_practice).astype(float),  # numerical
    'loan_amount':    np.random.randint(5000, 80000, n_practice).astype(float),   # numerical
    'employment':     np.random.choice(['salaried', 'self_employed', 'unemployed'], n_practice),  # nominal
    'risk_level':     np.random.choice(['low', 'medium', 'high'], n_practice),    # ordinal
})

# Target: approved based on credit score and income
practice_df['approved'] = ((practice_df['credit_score'] > 650) & 
                            (practice_df['annual_income'] > 50000)).astype(int)

print("Practice dataset:")
print(practice_df.head(8).to_string(index=False))
print(f"\nShape: {practice_df.shape}")
print(f"Approval rate: {practice_df['approved'].mean()*100:.1f}%")
print()
print("Your tasks:")
print("  1. Which scaler for credit_score, annual_income, loan_amount?")
print("  2. Which encoding for employment?")
print("  3. Which encoding for risk_level? What is the correct order?")
print("  4. Build the Pipeline, train it, and print accuracy.")
print("  5. Save and reload the pipeline. Predict for a new applicant.")

# --- Your solution below ---


In [ ]:
# SOLUTION — try on your own first before looking here

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

num_cols  = ['credit_score', 'annual_income', 'loan_amount']
nom_cols  = ['employment']
ord_cols  = ['risk_level']
risk_order = ['low', 'medium', 'high']   # correct ordinal order

preproc = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('nom', OneHotEncoder(drop='first', sparse_output=False), nom_cols),
    ('ord', OrdinalEncoder(categories=[risk_order]), ord_cols),
])

pipe = Pipeline([
    ('preproc', preproc),
    ('model',   KNeighborsClassifier(n_neighbors=7))
])

X_p = practice_df[num_cols + nom_cols + ord_cols]
y_p = practice_df['approved'].values
X_tr, X_te, y_tr, y_te = train_test_split(X_p, y_p, test_size=0.2, random_state=42)

pipe.fit(X_tr, y_tr)
print(f"Loan approval pipeline accuracy: {accuracy_score(y_te, pipe.predict(X_te))*100:.1f}%")

joblib.dump(pipe, 'day11_loan_pipeline.pkl')
loaded = joblib.load('day11_loan_pipeline.pkl')

new_applicant = pd.DataFrame({
    'credit_score':  [720],
    'annual_income': [85000],
    'loan_amount':   [30000],
    'employment':    ['salaried'],
    'risk_level':    ['low']
})

result = loaded.predict(new_applicant)
print(f"New applicant prediction: {'APPROVED' if result[0]==1 else 'REJECTED'}")

---
## What's Next

**Day 12: How ML Actually Works**  
The learn-from-data loop. Parameters, loss functions, and optimization. No black box.

---
**GitHub repo:** https://github.com/VaishnaviJagtap18/42-Days-of-ML-Challenge  
**LinkedIn:** Follow for Day 12 tomorrow  
#42DaysOfML #MachineLearning #Python #MLEngineer